In [ ]:
from pathlib import Path

print("Train images:", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\images\train").glob("*.jpg"))))
print("Train labels:", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_)FOG\labels\train").glob("*.txt"))))

print("Val images:", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\images\val").glob("*.jpg"))))
print("Val labels:", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\labels\val").glob("*.txt"))))

In [ ]:
from pathlib import Path

train_imgs = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\images\train").glob("*.jpg")}
train_lbls = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\labels\train").glob("*.txt")}

print("Train matched:", len(train_imgs & train_lbls))

val_imgs = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\images\val").glob("*.jpg")}
val_lbls = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\labels\val").glob("*.txt")}

print("Val matched:", len(val_imgs & val_lbls))

In [ ]:
import os
import xml.etree.ElementTree as ET

# PATHS

xml_folder = r"D:\DMSc_Dissertation_Project\datasets\DAWN\Fog\Fog\Fog\Fog_PASCAL_VOC"

label_folder = r"D:\DMSc_Dissertation_Project\datasets\DAWN\Fog\Fog\Fog\Fog_YOLO_FIXED"

os.makedirs(label_folder, exist_ok=True)

# CLASS MAPPING

class_map = {
    "person": 0,
    "rider": 1,
    "car": 2,
    "truck": 3,
    "bus": 4,
    "train": 5,
    "motorcycle": 6,
    "bicycle": 7
}

# CONVERT XML TO YOLO

count = 0

for file in os.listdir(xml_folder):

    if not file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(xml_folder, file))
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    txt_file = os.path.join(
        label_folder,
        file.replace(".xml", ".txt")
    )

    with open(txt_file, "w") as f:

        for obj in root.findall("object"):

            name = obj.find("name").text.lower()

            if name not in class_map:
                continue

            cls = class_map[name]

            box = obj.find("bndbox")

            xmin = float(box.find("xmin").text)
            ymin = float(box.find("ymin").text)
            xmax = float(box.find("xmax").text)
            ymax = float(box.find("ymax").text)

            x = ((xmin + xmax) / 2) / width
            y = ((ymin + ymax) / 2) / height
            w = (xmax - xmin) / width
            h = (ymax - ymin) / height

            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

    count += 1

print(f"Conversion completed successfully.")
print(f"XML files converted: {count}")
print(f"Labels saved in:\n{label_folder}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")

model.train(
    data=r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG\dawn_fog.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    workers=4,
    device=0,
    cache=False,
    amp=True,
    project="DAWN_FOG_Project",
    name="train_yolov8l_fog"
)

In [ ]:
import os
import shutil
import pandas as pd

# 1. Configuration
CONDITION = "fog"
PROJECT_DIR = r"C:\Users\Varis\runs\detect\DAWN_FOG_Project\train_yolov8l_fog-4"
BACKUP_DIR = r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG"

# 2. Ensure backup directory exists
os.makedirs(BACKUP_DIR, exist_ok=True)

# 3. Files to backup
files_to_copy = [
    "results.csv", "results.png", "confusion_matrix.png", 
    "confusion_matrix_normalized.png", "PR_curve.png", "P_curve.png", 
    "R_curve.png", "F1_curve.png", "labels.jpg", 
    "labels_correlogram.jpg", "args.yaml"
]

# Copy Weights (best.pt and last.pt)
shutil.copy2(os.path.join(PROJECT_DIR, "weights", "best.pt"), BACKUP_DIR)
shutil.copy2(os.path.join(PROJECT_DIR, "weights", "last.pt"), BACKUP_DIR)

# Copy Diagnostic Files
for file in files_to_copy:
    src_path = os.path.join(PROJECT_DIR, file)
    if os.path.exists(src_path):
        shutil.copy2(src_path, BACKUP_DIR)
    else:
        # Check inside the 'plots' subfolder if not found in root
        plots_path = os.path.join(PROJECT_DIR, "plots", file)
        if os.path.exists(plots_path):
            shutil.copy2(plots_path, BACKUP_DIR)
        else:
            print(f"Warning: {file} not found in project directory.")

# 4. Create Performance Summary CSV
results_df = pd.read_csv(os.path.join(PROJECT_DIR, "results.csv"))
results_df.columns = results_df.columns.str.strip()
summary_df = pd.DataFrame([{
    "Condition": CONDITION.upper(),
    "Max_mAP50": results_df['metrics/mAP50(B)'].max(),
    "Max_mAP50-95": results_df['metrics/mAP50-95(B)'].max()
}])
summary_df.to_csv(os.path.join(BACKUP_DIR, "performance_summary.csv"), index=False)

# 5. Create Dissertation Info File
with open(os.path.join(BACKUP_DIR, "training_info.txt"), "w") as f:
    f.write(f"Dissertation Project Backup: {CONDITION.upper()}\n")
    f.write(f"Path: {PROJECT_DIR}\n")
    f.write(f"Final mAP50: {results_df['metrics/mAP50(B)'].max():.4f}\n")
    f.write("All graphs and weights successfully backed up to D: drive.")

print(f"Professional backup for {CONDITION.upper()} completed successfully at: {BACKUP_DIR}")

In [ ]:
import os
import shutil

# Paths
SOURCE_DIR = r"C:\Users\Varis\runs\detect\DAWN_FOG_Project\train_yolov8l_fog-4"
BACKUP_DIR = r"D:\DMSc_Dissertation_Project\datasets\DAWN\YOLO_FOG"

# Files to find
target_files = [
    "PR_curve.png", "P_curve.png", "R_curve.png", 
    "F1_curve.png", "labels.jpg", "labels_correlogram.jpg"
]

print("Scanning for missing files...")

for root, dirs, files in os.walk(SOURCE_DIR):
    for file in files:
        if file in target_files:
            src_path = os.path.join(root, file)
            dest_path = os.path.join(BACKUP_DIR, file)
            shutil.copy2(src_path, dest_path)
            print(f"Found and copied: {file}")

print(f"\nAll available visual files have been moved to: {BACKUP_DIR}")